In [ ]:
#Aimen Fatima 23i-7716
#Assignment 3

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [57]:
from sklearn.model_selection import train_test_split , GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

In [28]:
df = pd.read_csv("/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv", 
                 encoding="ISO-8859-1", 
                 header=None,
                 names=["target", "id", "date", "flag", "user", "text"])

# Display the DataFrame
print(df.head())

   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [32]:
df['target'].unique()

array([0, 4])

In [ ]:
                                                  # PART 1 #
############# Preprocessing ##############
# 1. The data is very large, it is taking alot of time to preprocess , so I have only taken 2000 rows from it.
#    1000 of each class
# 2. Basic preprocessing like removing URLs, stop words , special characters and making the text lowercase  
# 3. Also changing the labels which are 0, 4 to 0,1 for ease

In [31]:
# Select 1000 rows from the start
df_start = df.head(1000)

# Select 1000 rows from the end
df_end = df.tail(1000)

# Concatenate the two parts
df = pd.concat([df_start, df_end])

In [34]:
df['target'].unique()

array([0, 4])

In [3]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
def preprocess_text(text):
    # Convert text to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize the text
    tokens = word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    # Join tokens back into a single string
    processed_text = ' '.join(tokens)
    return processed_text

In [35]:
df['text'] = df['text'].apply(preprocess_text)

In [44]:
df['target'] = df['target'].map({4: 1, 0: 0}) #postive is 4 which im making 1 and negative is 0 which stays the same 

In [68]:
                                               # PART 2 #
######### Training the models################

In [45]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=42)


In [46]:
# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [47]:
# Train GBM model
gbm_model = GradientBoostingClassifier()
gbm_model.fit(X_train_tfidf, y_train)

GradientBoostingClassifier()

In [48]:
# Train AdaBoost model
adaboost_model = AdaBoostClassifier()
adaboost_model.fit(X_train_tfidf, y_train)

AdaBoostClassifier()

In [49]:
# Predictions
gbm_preds = gbm_model.predict(X_test_tfidf)
adaboost_preds = adaboost_model.predict(X_test_tfidf)

In [50]:
# Evaluate models
gbm_accuracy = accuracy_score(y_test, gbm_preds)
adaboost_accuracy = accuracy_score(y_test, adaboost_preds)

In [51]:
gbm_f1 = f1_score(y_test, gbm_preds)
adaboost_f1 = f1_score(y_test, adaboost_preds)

In [52]:
print("GBM Accuracy:", gbm_accuracy)
print("AdaBoost Accuracy:", adaboost_accuracy)   #Gradient Boosting Machine (GBM) model outperforms the AdaBoost model in terms of accuracy, with GBM 

GBM Accuracy: 0.665
AdaBoost Accuracy: 0.6025


In [53]:
print("GBM F1 Score:", gbm_f1)
print("AdaBoost F1 Score:", adaboost_f1) # These scores suggest that GBM performs slightly better in terms of precision and recall balance compared to AdaBoost

GBM F1 Score: 0.7048458149779736
AdaBoost F1 Score: 0.6851485148514851


In [ ]:
                                                     # PART 3 #
########## Trying to Improve results #############
# Using Grid-search

In [55]:
gbm_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}
adaboost_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2]
}

In [58]:
gbm_grid_search = GridSearchCV(gbm_model, gbm_param_grid, cv=3, scoring='accuracy')
gbm_grid_search.fit(X_train_tfidf, y_train)

# Get the best GBM model
best_gbm_model = gbm_grid_search.best_estimator_

In [59]:
adaboost_grid_search = GridSearchCV(adaboost_model, adaboost_param_grid, cv=3, scoring='accuracy')
adaboost_grid_search.fit(X_train_tfidf, y_train)

# Get the best AdaBoost model
best_adaboost_model = adaboost_grid_search.best_estimator_

In [60]:
# Make predictions
gbm_preds = best_gbm_model.predict(X_test_tfidf)
adaboost_preds = best_adaboost_model.predict(X_test_tfidf)

In [62]:
gbm_accuracy = accuracy_score(y_test, gbm_preds)
adaboost_accuracy = accuracy_score(y_test, adaboost_preds)

In [63]:
gbm_f1 = f1_score(y_test, gbm_preds)
adaboost_f1 = f1_score(y_test, adaboost_preds)

In [64]:
print("Best GBM Model:", best_gbm_model)
print("GBM Accuracy:", gbm_accuracy)
print("GBM F1 Score:", gbm_f1)

Best GBM Model: GradientBoostingClassifier(max_depth=5, n_estimators=200)
GBM Accuracy: 0.68
GBM F1 Score: 0.7050691244239631


In [65]:
print("Best AdaBoost Model:", best_adaboost_model)
print("AdaBoost Accuracy:", adaboost_accuracy)
print("AdaBoost F1 Score:", adaboost_f1)

Best AdaBoost Model: AdaBoostClassifier(learning_rate=0.2, n_estimators=200)
AdaBoost Accuracy: 0.6925
AdaBoost F1 Score: 0.7248322147651006


In [67]:
## both the accuracies have increased 
#GBM from 66% to 68%
#ADABoost from 60% to 69% 